# Módulo 8 — Transformada de Fourier
## Seção 8.2 — Frequência, amplitude, fase e período

> **Estado editorial:** material didático fundamentado nas páginas indicadas; exercícios e exemplos numéricos são desenvolvimentos próprios. Python básico é suficiente. Dados sintéticos.

**Referência principal consultada:** Morettin, P. A.; Toloi, C. M. C. *Análise de séries temporais: modelos lineares univariados*, vol. 1, 3ª ed. (2018), cap. 13, §13.1, p. 335 e §13.2, pp. 335–336 (equações 13.1–13.5); apoio: §2.7.1, pp. 41–43 (harmônicas em modelos determinísticos) e §3.3, pp. 68–74 (sazonalidade). Nas seções seguintes do módulo serão estudados números complexos, séries de Fourier, DFT e FFT.

**Limite bibliográfico:** o capítulo 13 fundamenta o modelo harmônico; os desenvolvimentos pedagógicos sobre sinais contínuos, frequência de amostragem e exemplos simulados são nossos, não transcrições do livro. O mapeamento de videoaulas do projeto classifica Fourier como *mencionado*, não desenvolvido; portanto, não usamos as videoaulas como demonstração matemática.

### Objetivos e pré-requisitos

Interpretar $R$, $T$, $f$, $\omega$ e $\phi$; converter unidades; calcular fase e amplitude a partir de $A,B$; estimar coeficientes com mínimos quadrados em um modelo de frequência conhecida; compreender aliasing introdutoriamente. Pré-requisito direto: Seção 8.1.

### 1. O problema: separar 'quão rápido', 'quão forte' e 'quando'

Duas oscilações podem ter o mesmo período e amplitudes diferentes, ou a mesma amplitude e estar deslocadas no tempo. O modelo do livro (§13.2, pp. 335–336, eq. 13.1) expressa esses atributos por

$$Z_t=\mu+R\cos(\omega t+\phi)+\varepsilon_t.$$

Aqui, $\mu$ é o nível médio determinístico, $R\ge0$ a amplitude, $\omega$ a frequência angular, $\phi$ a fase e $\varepsilon_t$ a parte aleatória. O parâmetro $t$ é o índice de tempo; devemos declarar se sua unidade é dia, mês, hora etc.

### 2. Frequência e período

**Frequência comum:** $f$ ciclos por unidade de tempo; **período:** $T$ unidades de tempo por ciclo; **frequência angular:** $\omega$ radianos por unidade de tempo. As identidades são

$$\boxed{f=\frac1T,\quad\omega=2\pi f=\frac{2\pi}{T},\quad T=\frac{2\pi}{\omega}}$$

para $f,\omega>0$. A equivalência deriva de que uma volta completa corresponde a $2\pi$ radianos.

**Exemplo numérico:** uma oscilação de período 12 meses tem $f=1/12$ ciclo/mês e $\omega=\pi/6$ rad/mês; em um ano realiza um ciclo. Uma oscilação de período 4 meses tem $f=1/4$ ciclo/mês e $\omega=\pi/2$ rad/mês. Quando $t$ está em meses, $\omega$ NÃO está em hertz (Hz, ciclos por segundo).

### 3. Amplitude e fase

O máximo de $s_t=\mu+R\cos(\omega t+\phi)$ é $\mu+R$ e o mínimo é $\mu-R$, no sinal contínuo e supondo $R\ge0$; em amostras discretas o máximo observado pode não alcançar o máximo teórico. O deslocamento em tempo é $\tau=-\phi/\omega$ (sinal da convenção $+\phi$), módulo $T$.

**Exemplo:** $s_t=4+2\cos(\frac\pi2t-\frac\pi2)=4+2\sin(\frac\pi2t)$. O máximo ocorre em $t=1+4k$, para inteiros $k$; $T=4$, $R=2$, $\mu=4$, $\phi=-\pi/2$. Sua fase desloca o pico uma unidade à direita em relação a $4+2\cos(\pi t/2)$.

### 4. Identidade trigonométrica e parametrização linear (livro, eqs. 13.1–13.3)

Usando $\cos(a+b)=\cos a\cos b-\sin a\sin b$:

$$\begin{aligned}
R\cos(\omega t+\phi)&=R\cos\phi\cos(\omega t)-R\sin\phi\sin(\omega t)\\
&=A\cos(\omega t)+B\sin(\omega t),
\end{aligned}$$

com $A=R\cos\phi$ e $B=-R\sin\phi$. Segue que

$$\boxed{R=\sqrt{A^2+B^2},\qquad\phi=\operatorname{atan2}(-B,A).}$$

A função `atan2(y,x)` determina o quadrante corretamente. Ela equivale ao tratamento por casos no livro (eq. 13.3, p. 336), módulo $2\pi$. Se $A=B=0$, amplitude nula e fase **indeterminada**, não zero por definição.

**Exemplo numérico:** $A=3$, $B=-4$ dão $R=5$ e $\phi=\operatorname{atan2}(4,3)\approx0,9273$ rad. Em particular, a fórmula incorreta $\arctan(-B/A)$ falha em quadrantes quando $A<0$.

### 5. O problema de mínimos quadrados: frequência conhecida

Morettin e Toloi (§13.2.1, p. 336) propõem minimizar

$$S(\mu,A,B)=\sum_{t=1}^N[Z_t-\mu-A\cos(\omega t)-B\sin(\omega t)]^2.$$

Seja $c_t=\cos(\omega t)$, $s_t=\sin(\omega t)$ e $r_t=Z_t-\mu-Ac_t-Bs_t$. Derivando termo a termo:

$$\frac{\partial S}{\partial\mu}=-2\sum r_t,\quad \frac{\partial S}{\partial A}=-2\sum c_t r_t,\quad\frac{\partial S}{\partial B}=-2\sum s_t r_t.$$

Igualando a zero, obtemos precisamente as equações normais (13.5), p. 336. Definindo

$$X=\begin{bmatrix}1&c_1&s_1\\\vdots&\vdots&\vdots\\1&c_N&s_N\end{bmatrix},\quad \theta=\begin{bmatrix}\mu\\A\\B\end{bmatrix},\quad z=\begin{bmatrix}Z_1\\\vdots\\Z_N\end{bmatrix},$$

as equações normais são $X^\top X\widehat\theta=X^\top z$. Se $X$ tem posto 3, $\widehat\theta=(X^\top X)^{-1}X^\top z$; na prática usamos `np.linalg.lstsq`, sem formar a inversa. **Ressalva:** sem posto completo, os parâmetros não são individualmente identificáveis, por exemplo em frequências degeneradas da amostragem.

### 6. Exemplo manual de estimação sem ruído

Considere $t=0,1,2,3$ e $Z_t=2+3\cos(\pi t/2)-\sin(\pi t/2)$. Então $z=(5,1,-1,3)$ e $\omega=\pi/2$. Como os cossenos e senos somam zero no ciclo completo,

$$\widehat\mu=\frac{5+1-1+3}{4}=2,$$

$$\widehat A=\frac{2}{4}\sum_{t=0}^3 Z_t\cos(\pi t/2)=\frac12(5+1)=3,$$

$$\widehat B=\frac{2}{4}\sum_{t=0}^3 Z_t\sin(\pi t/2)=\frac12(1-3)=-1.$$

Logo, $\widehat R=\sqrt{10}\approx3,1623$ e $\widehat\phi=\operatorname{atan2}(1,3)\approx0,32175$ rad. A simplificação $2/N$ dos coeficientes decorre da ortogonalidade *nesta grade completa*; **não vale para quaisquer instantes ou frequências**.

### 7. Laboratório 1 — efeito de cada parâmetro



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 24, 1200)
fig, axs = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
for amp in [1, 2, 4]: axs[0].plot(t, amp*np.cos(2*np.pi*t/12), label=f'R={amp}')
for periodo in [4, 6, 12]: axs[1].plot(t, 2*np.cos(2*np.pi*t/periodo), label=f'T={periodo}')
for fase in [0, -np.pi/2, np.pi/2]: axs[2].plot(t, 2*np.cos(2*np.pi*t/12+fase), label=f'fase={fase:.2f}')
for ax in axs: ax.legend(); ax.grid(alpha=.2)
axs[0].set_title('Amplitude'); axs[1].set_title('Período'); axs[2].set_title('Fase')
axs[2].set_xlabel('Tempo (meses)')
fig.suptitle('Figura 8.2.1 — Parâmetros de uma senoide')
plt.tight_layout(); plt.show()



**Interpretação:** alterar $R$ modifica a altura da oscilação, alterar $T$ modifica o número de ciclos no mesmo horizonte, alterar $\phi$ desloca os picos sem modificar $R$ ou $T$.

### 8. Laboratório 2 — estimar amplitude e fase a partir de observações



In [ ]:
rng = np.random.default_rng(2026)
t = np.arange(120)
omega = 2*np.pi/12
mu, R, fase = 10., 3., -np.pi/3
z = mu + R*np.cos(omega*t+fase) + rng.normal(0, 0.6, len(t))
X = np.column_stack([np.ones_like(t), np.cos(omega*t), np.sin(omega*t)])
coef, residuos_soma_quadrados, posto, singulares = np.linalg.lstsq(X, z, rcond=None)
mu_hat, A_hat, B_hat = coef
R_hat = np.hypot(A_hat, B_hat)
fase_hat = np.arctan2(-B_hat, A_hat)
ajustado = X @ coef
print('posto:', posto, 'mu:', round(mu_hat,3), 'R:', round(R_hat,3), 'fase:', round(fase_hat,3))
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(t, z, '.', alpha=.5, label='observado')
ax.plot(t, ajustado, '-', label='harmônico estimado')
ax.set(xlabel='Mês', ylabel='Valor', title='Figura 8.2.2 — Ajuste harmônico por MQ')
ax.legend(); plt.show()
assert posto == 3



**Interpretação:** a estimação recupera os parâmetros aproximados quando a frequência está correta, a amostra contém ciclos suficientes e o ruído é moderado. Não se espera igualdade exata em presença de ruído. Se a frequência for desconhecida, estimar apenas $A,B$ não resolve todo o problema; esse tema aparece no §13.2.2, pp. 340–345, e será aprofundado no Módulo 9.

### 9. Amostragem e aliasing: uma antecipação rigorosa

Se medimos um sinal contínuo $s(t)=\cos(2\pi f t+\phi)$ nos instantes $t=n\Delta$, temos $s[n]=\cos(2\pi(f/f_s)n+\phi)$, com $f_s=1/\Delta$. Para qualquer inteiro $k$,

$$\cos[2\pi(f+k f_s)n/f_s+\phi]=\cos(2\pi fn/f_s+\phi+2\pi kn)=s[n].$$

Portanto, frequências contínuas separadas por múltiplos de $f_s$ produzem as mesmas amostras. Esse é o mecanismo básico do **aliasing**. Para reconstruir sem ambiguidade um sinal ideal limitado em banda, a condição de Nyquist exige amostragem estritamente acima de duas vezes a maior frequência presente, além de hipóteses apropriadas sobre o sinal e a reconstrução. O tema será desenvolvido em 8.8.

### 10. Limitações

- A fase é definida módulo $2\pi$; comparações exigem convenção de sinal e origem do tempo.
- Em frequência zero, a componente cosseno é indistinguível do intercepto; na frequência de Nyquist, certas componentes seno degeneram na grade inteira.
- Frequência conhecida é hipótese substantiva; um erro na frequência provoca ajuste sistematicamente inadequado.
- Ruído autocorrelacionado afeta a inferência usual de MQ, mesmo que os coeficientes sejam computáveis.
- Uma boa curva ajustada não demonstra causalidade nem que existam ciclos permanentes no processo gerador.

### 11. Exercícios para desenvolver autonomia

1. Calcule $f$ e $\omega$ para períodos de 7 dias e 24 horas; declare as unidades.
2. Para $A=-3$, $B=-4$, obtenha $R$ e $\phi$ com o quadrante correto; compare `atan2` e `atan`.
3. Demonstre por que o máximo contínuo de $\mu+R\cos(\omega t+\phi)$ é $\mu+R$, para $R\ge0$, mas pode não ocorrer na grade amostral.
4. Monte, no papel, a matriz $X$ do exemplo manual de quatro observações e verifique $X^\top X$.
5. Escreva um pseudocódigo para estimar $R$ e $\phi$ a partir de uma frequência conhecida; implemente em seguida sem olhar o laboratório 2.
6. Experimente ajustar um período 10 aos dados gerados com período 12; compare resíduos, sem concluir apenas pelo gráfico.

**Próxima seção:** números complexos e a fórmula de Euler, necessários para representar exponenciais complexas e deduzir a transformada de Fourier.
